# Billing QLoRA with Unsloth (Chapter 43)

An experiment tool, not the pipeline: anything that works here becomes a script (Chapter 26). **Only data with no customer content belongs in a free notebook**: the synthetic set, the support team's exemplars and the replay set (Chapter 43, Topic 1). Run the cells top to bottom on a free GPU runtime.

## 1. What did we get?
The first cell prints the GPU's name and memory: what a free session assigns varies, and an unexpected smaller GPU explains most out-of-memory surprises. Older cards have no bfloat16, so the compute type is chosen from what the GPU reports.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Runtime > Change runtime type > a GPU'
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')
bf16_ok = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16      # older cards: float16
print('bf16' if bf16_ok else 'fp16')

## 2. Install the repository's pinned versions
The versions in `training/uv.lock` at this tag, not the latest. Restart the runtime if pip asks you to.

In [ ]:
%pip install -q 'unsloth==2026.10.9' 'unsloth_zoo==2026.10.9' 'bitsandbytes==0.50.2' 'transformers==4.57.6' 'trl==0.29.1' 'peft==0.21.2' 'accelerate==1.15.0' 'datasets==4.8.5' 'mlflow'

## 3. Survive disconnects
Checkpoints go to storage that outlives the session. A free session is time-limited and can disconnect; `has_checkpoint` lets the next run resume from the last one.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ledgerly-runs/billing-qlora'

import glob


def has_checkpoint(path):
    found = sorted(glob.glob(f'{path}/checkpoint-*'))
    return found[-1] if found else None

## 4. Only non-customer data in a notebook
The private Hub repository `dataset:publish --hub-private=...` wrote (exemplars, synthetic and replay only), at a pinned revision. Put a read-only token in Colab's secrets as `HF_TOKEN`; never paste one into a cell.

In [ ]:
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata

HUB_REPO = 'ledgerly/billing-synthetic'       # your own private repository, from dataset:publish
REVISION = 'a41c09e'                          # pinned: the revision dataset:publish printed
token = userdata.get('HF_TOKEN')
parts = [load_dataset(HUB_REPO, data_files=name, revision=REVISION, token=token, split='train') for name in ('synthetic.jsonl', 'exemplars.jsonl', 'replay.jsonl')]
rows = concatenate_datasets(parts).shuffle(seed=42)
split = rows.train_test_split(test_size=0.08, seed=42)
print(len(split['train']), 'train rows,', len(split['test']), 'validation rows')

## 5. QLoRA with Unsloth
Import `unsloth` before `trl` and `transformers` so its patches apply. The adapter it saves is a normal `peft` adapter, and Chapter 42's export handles it unchanged.

In [ ]:
from unsloth import FastLanguageModel                 # before trl and transformers
from trl import SFTConfig, SFTTrainer

model, tok = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Llama-3.1-8B-Instruct', max_seq_length=2048, load_in_4bit=True, token=token,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',             # its memory-saving checkpointing
    random_state=42,
)


def render(row):
    prompt = tok.apply_chat_template(row['prompt'], tokenize=False, add_generation_prompt=True)
    text = tok.apply_chat_template(row['prompt'] + row['completion'], tokenize=False)
    return {'prompt': prompt, 'completion': text[len(prompt):]}


train = split['train'].map(render, remove_columns=split['train'].column_names)
val = split['test'].map(render, remove_columns=split['test'].column_names)

trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=train, eval_dataset=val,
                     args=SFTConfig(output_dir=OUT, num_train_epochs=2, learning_rate=2e-4,
                                    per_device_train_batch_size=2, gradient_accumulation_steps=8,
                                    fp16=not bf16_ok, bf16=bf16_ok, save_steps=200, report_to='none'))
trainer.train(resume_from_checkpoint=has_checkpoint(OUT))

## 6. Save the adapter before the session ends
To storage that survives it. From here the adapter goes through the same export and checks as every other (Chapter 42, Topic 7).

In [ ]:
model.save_pretrained(f'{OUT}/adapter')                # a standard peft adapter
print('saved to', f'{OUT}/adapter')